In [1]:
import numpy as np
from datetime import datetime, timezone

# -------------------------------
# Constants
# -------------------------------
R_EARTH = 6378137.0          # meters (WGS84)
f = 1 / 298.257223563        # flattening
e2 = f * (2 - f)             # eccentricity squared

# -------------------------------
# Julian Date
# -------------------------------
def julian_date(dt):
    """Convert datetime to Julian Date."""
    dt = dt.astimezone(timezone.utc)
    year = dt.year
    month = dt.month
    day = dt.day + (dt.hour + dt.minute/60 + dt.second/3600)/24

    if month <= 2:
        year -= 1
        month += 12

    A = int(year / 100)
    B = 2 - A + int(A / 4)

    JD = int(365.25 * (year + 4716)) + int(30.6001 * (month + 1)) + day + B - 1524.5
    return JD

# -------------------------------
# GMST (Greenwich Mean Sidereal Time)
# -------------------------------
def gmst(dt):
    JD = julian_date(dt)
    T = (JD - 2451545.0) / 36525.0

    GMST = 280.46061837 + 360.98564736629 * (JD - 2451545.0) \
           + 0.000387933 * T**2 - (T**3) / 38710000.0

    return np.radians(GMST % 360)

# -------------------------------
# Geodetic → ECEF
# -------------------------------
def geodetic_to_ecef(lat, lon, h):
    lat = np.radians(lat)
    lon = np.radians(lon)

    N = R_EARTH / np.sqrt(1 - e2 * np.sin(lat)**2)

    x = (N + h) * np.cos(lat) * np.cos(lon)
    y = (N + h) * np.cos(lat) * np.sin(lon)
    z = (N * (1 - e2) + h) * np.sin(lat)

    return np.array([x, y, z])

In [2]:
# -------------------------------
# ECEF → ENU
# -------------------------------
def ecef_to_enu(r, lat, lon):
    lat = np.radians(lat)
    lon = np.radians(lon)

    transform = np.array([
        [-np.sin(lon),  np.cos(lon), 0],
        [-np.sin(lat)*np.cos(lon), -np.sin(lat)*np.sin(lon), np.cos(lat)],
        [ np.cos(lat)*np.cos(lon),  np.cos(lat)*np.sin(lon), np.sin(lat)]
    ])

    return transform @ r

# -------------------------------
# Main: ECEF → Alt-Az
# -------------------------------
def ecef_to_altaz(r_ecef, obs_lat, obs_lon, obs_h, dt):
    """
    r_ecef: target position (meters)
    obs_lat, obs_lon: degrees
    obs_h: meters
    dt: datetime (UTC)
    """

    # Observer ECEF
    r_obs = geodetic_to_ecef(obs_lat, obs_lon, obs_h)

    # Relative vector
    rho = r_ecef - r_obs

    # Convert to ENU
    enu = ecef_to_enu(rho, obs_lat, obs_lon)
    E, N, U = enu

    # Range
    r = np.linalg.norm(enu)

    # Altitude
    alt = np.arcsin(U / r)

    # Azimuth
    az = np.arctan2(E, N)

    # Convert to degrees
    alt_deg = np.degrees(alt)
    az_deg = np.degrees(az) % 360

    return alt_deg, az_deg



In [ ]:
# -------------------------------
# Usage
# -------------------------------
if __name__ == "__main__":
    # Example satellite ECEF (meters)
    r_ecef = np.array([1.5e7, 2.0e7, 2.1e7])

    # Observer location
    lat = 17.3850       # Hyderabad approx
    lon = 78.4867
    h = 500             # meters

    # Time (UTC)
    dt = datetime(2026, 5, 2, 12, 0, 0, tzinfo=timezone.utc)

    alt, az = ecef_to_altaz(r_ecef, lat, lon, h, dt)

    print(f"Altitude: {alt:.2f} deg")
    print(f"Azimuth : {az:.2f} deg")

Altitude: 51.49 deg
Azimuth : 321.17 deg
